# In-hospital mortality with GPT2-AR (autoregressive rollout + linear classifier)

Reconstruction of `transformers/models_nesy.py` (`CustomGPT` with LM head), `pretrain/trainer.py` and
`transformers/trainer_linear.py` / `mortality_stability.py` for the MIMIC-IV mortality task.

**Stage 1: next-event pretraining.** GPT2 (`GPT2LMHeadModel`) with a new event vocabulary and an additive
time embedding (`token_emb + time_layer(t)`) is trained with causal LM loss on the training stays.

**Stage 2: AR classifier.** For each stay the model *rolls out* `N_STEPS = 3` future events: it samples the next
event, appends it with time `t_last + 0.5 h`, and repeats. The next-event logits at each step go through a shared
`Linear(vocab_size, 1)`. The mortality logit is the **max over the steps**, i.e. "does the predicted near-future
trajectory contain high-risk events?". Only the last 2 GPT2 blocks and the classifier are fine-tuned
(`full=False` in the original).

Differences from the original scripts:
* The next-event logits are read at the last *real* position of each sequence, not at `[:, -1]`, which is padding for all but the longest sequence in a batch.
* Evaluation uses greedy decoding (deterministic), training uses sampling (`temperature=1`).
* The pretraining stage is included here instead of loading the external `checkpoints/gpt_eventval.pth`.

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

sys.path.insert(0, str(Path.cwd()))
from mortality_common import (
    MIMIC_DIR, CHECKPOINT_DIR, TIME_SCALE, set_seed, get_device, load_tokenizer, make_loaders, encode_batch,
    autocast_context, make_grad_scaler, count_parameters, tokenizer_fingerprint, train_binary_classifier, save_results,
)

In [ ]:
# ---------------- configuration ----------------
SEED = 1234
GPU = 0
PRETRAINED = 'gpt2'
N_DEMO = 3
N_STEPS = 3                  # future events rolled out per stay (self.N in trainer_linear.py)
STEP_HOURS = 0.5             # time increment of each generated event
CONTEXT_LENGTH = 1024        # GPT2 max positions; room is left for the N_STEPS - 1 generated events
MAX_EVENTS = CONTEXT_LENGTH - N_DEMO - (N_STEPS - 1)

# stage 1: next-event pretraining
PRETRAIN_EPOCHS = 3
PRETRAIN_LR = 5e-5
PRETRAIN_BATCH_SIZE = 8
PRETRAIN_GRAD_ACCUM = 4
PRETRAIN_CKPT = CHECKPOINT_DIR / 'gpt2_lm_pretrain.pth'   # reused if it exists and was trained with the current tokenizer

# stage 2: AR classifier
N_EPOCHS = 10
PATIENCE = 3
LEARNING_RATE = 2e-5
BATCH_SIZE = 8
GRAD_ACCUM = 2
N_TRAINABLE_BLOCKS = 2

set_seed(SEED)
device = get_device(GPU)
tokenizer = load_tokenizer()
PAD_ID = tokenizer.pad_token_id
loaders = make_loaders(BATCH_SIZE, max_events=MAX_EVENTS, eval_batch_size=2 * BATCH_SIZE)
print('device:', device, '| vocab:', len(tokenizer))

## Model

In [ ]:
from transformers import GPT2LMHeadModel


class GPT2EventLM(nn.Module):
    """GPT2 language model over event tokens with an additive time embedding (models_nesy.CustomGPT)."""

    def __init__(self, vocab_size, pretrained='gpt2'):
        super().__init__()
        self.model = GPT2LMHeadModel.from_pretrained(pretrained)
        self.model.resize_token_embeddings(vocab_size)
        hidden_size = self.model.config.n_embd
        self.time_layer = nn.Linear(1, hidden_size)

    def freeze_except_last_blocks(self, n_blocks):
        for p in self.model.parameters():
            p.requires_grad = False
        for block in self.model.transformer.h[-n_blocks:]:
            for p in block.parameters():
                p.requires_grad = True
        for p in self.model.transformer.ln_f.parameters():
            p.requires_grad = True

    def hidden_states(self, input_ids, times, attention_mask):
        embs = self.model.transformer.wte(input_ids) + self.time_layer(times.unsqueeze(-1))
        return self.model.transformer(inputs_embeds=embs, attention_mask=attention_mask).last_hidden_state

    def forward(self, input_ids, times, attention_mask):
        return self.model.lm_head(self.hidden_states(input_ids, times, attention_mask))  # [B, L, V]

    def next_event_logits(self, input_ids, times, attention_mask, positions):
        """LM logits only at `positions` (one per sequence): [B, V]."""
        hidden = self.hidden_states(input_ids, times, attention_mask)
        return self.model.lm_head(hidden[torch.arange(hidden.size(0), device=hidden.device), positions])


model = GPT2EventLM(len(tokenizer), PRETRAINED).to(device)
count_parameters(model)

## Stage 1: next-event pretraining on the training stays
Same objective as `pretrain/trainer.py`: cross-entropy on the next token. The demographic prefix and padding are
excluded from the targets.

In [ ]:
def lm_loss(model, batch):
    input_ids, times, attention_mask = encode_batch(batch, PAD_ID, device)
    logits = model(input_ids[:, :-1], times[:, :-1], attention_mask[:, :-1])
    targets = input_ids[:, 1:].clone()
    targets[attention_mask[:, 1:] == 0] = -100
    targets[:, : N_DEMO - 1] = -100  # positions predicting demographic tokens
    loss = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)
    valid = targets != -100
    correct = ((logits.argmax(-1) == targets) & valid).sum().item()
    return loss, correct, valid.sum().item()


@torch.no_grad()
def lm_evaluate(model, loader):
    model.eval()
    total_loss, correct, count = 0.0, 0, 0
    for batch in tqdm(loader, desc='lm eval', leave=False):
        with autocast_context(device):
            loss, c, n = lm_loss(model, batch)
        total_loss += loss.item() * n
        correct += c
        count += n
    return total_loss / count, correct / count


# The sidecar file records which tokenizer / base model the LM was trained with. Re-running 01 (e.g. with a different
# SUBSAMPLE_PATIENTS) rebuilds the vocabulary, and a stale checkpoint must not be reused.
PRETRAIN_META = PRETRAIN_CKPT.with_suffix('.json')
current_meta = {'tokenizer': tokenizer_fingerprint(), 'vocab_size': len(tokenizer), 'pretrained': str(PRETRAINED)}
saved_meta = json.loads(PRETRAIN_META.read_text()) if PRETRAIN_META.exists() else None
if PRETRAIN_CKPT.exists() and saved_meta != current_meta:
    print(f'ignoring {PRETRAIN_CKPT}: trained with {saved_meta}, current setup is {current_meta} -> pretraining again')

if PRETRAIN_CKPT.exists() and saved_meta == current_meta:
    print('loading pretrained LM from', PRETRAIN_CKPT)
    model.load_state_dict(torch.load(PRETRAIN_CKPT, map_location=device, weights_only=True))
else:
    PRETRAIN_CKPT.parent.mkdir(parents=True, exist_ok=True)
    PRETRAIN_META.unlink(missing_ok=True)
    lm_train_loader = torch.utils.data.DataLoader(
        loaders['train'].dataset, batch_size=PRETRAIN_BATCH_SIZE, shuffle=True, collate_fn=loaders['train'].collate_fn)
    optimizer = torch.optim.AdamW(model.parameters(), lr=PRETRAIN_LR, weight_decay=0.01)
    scaler = make_grad_scaler(device)
    best_val = float('inf')
    for epoch in range(1, PRETRAIN_EPOCHS + 1):
        model.train()
        running = 0.0
        progress = tqdm(lm_train_loader, desc=f'pretrain epoch {epoch}')
        for step, batch in enumerate(progress):
            with autocast_context(device):
                loss, _, _ = lm_loss(model, batch)
            scaler.scale(loss / PRETRAIN_GRAD_ACCUM).backward()
            if (step + 1) % PRETRAIN_GRAD_ACCUM == 0 or step == len(lm_train_loader) - 1:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            running += loss.item()
            progress.set_postfix(loss=f'{running / (step + 1):.4f}')
        val_loss, val_acc = lm_evaluate(model, loaders['val'])
        print(f'[pretrain epoch {epoch}] train_loss={running / len(lm_train_loader):.4f} val_loss={val_loss:.4f} val_next_event_acc={val_acc:.4f}')
        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), PRETRAIN_CKPT)
            PRETRAIN_META.write_text(json.dumps(current_meta))
    model.load_state_dict(torch.load(PRETRAIN_CKPT, map_location=device, weights_only=True))

## Stage 2: autoregressive rollout + linear classifier

In [ ]:
def ar_rollout(model, input_ids, times, attention_mask, n_steps, temperature):
    """Returns next-event logits for each of the n_steps rolled-out positions: [B, n_steps, V]."""
    batch_idx = torch.arange(input_ids.size(0), device=input_ids.device)
    lengths = attention_mask.sum(1)
    step_logits = []
    for step in range(n_steps):
        next_logits = model.next_event_logits(input_ids, times, attention_mask, lengths - 1)  # last real position
        step_logits.append(next_logits)
        if step == n_steps - 1:
            break
        with torch.no_grad():
            if temperature > 0:
                next_token = torch.multinomial(F.softmax(next_logits.float() / temperature, dim=-1), 1).squeeze(-1)
            else:
                next_token = next_logits.argmax(-1)
        # append the generated event (new tensors -> no in-place edits of tensors saved for backward)
        input_ids = F.pad(input_ids, (0, 1), value=PAD_ID)
        times = F.pad(times, (0, 1))
        attention_mask = F.pad(attention_mask, (0, 1))
        input_ids[batch_idx, lengths] = next_token
        times[batch_idx, lengths] = times[batch_idx, lengths - 1] + STEP_HOURS / TIME_SCALE
        attention_mask[batch_idx, lengths] = 1
        lengths = lengths + 1
    return torch.stack(step_logits, dim=1)


class GPT2ARClassifier(nn.Module):
    def __init__(self, lm, vocab_size, n_steps):
        super().__init__()
        self.lm = lm
        self.classifier = nn.Linear(vocab_size, 1)
        self.n_steps = n_steps

    def forward(self, input_ids, times, attention_mask, temperature=1.0):
        step_logits = ar_rollout(self.lm, input_ids, times, attention_mask, self.n_steps, temperature)
        return self.classifier(step_logits.float()).squeeze(-1).max(dim=-1).values  # max over rollout steps


model.freeze_except_last_blocks(N_TRAINABLE_BLOCKS)
ar_model = GPT2ARClassifier(model, len(tokenizer), N_STEPS).to(device)
count_parameters(ar_model)


def predict_fn(ar_model, batch, train):
    input_ids, times, attention_mask = encode_batch(batch, PAD_ID, device)
    return ar_model(input_ids, times, attention_mask, temperature=1.0 if train else 0.0)

In [ ]:
run_name = f'gpt2_ar_seed{SEED}'
history, test_metrics, test_pred = train_binary_classifier(
    ar_model, predict_fn, loaders, device, CHECKPOINT_DIR / f'{run_name}.pth',
    learning_rate=LEARNING_RATE, n_epochs=N_EPOCHS, patience=PATIENCE, grad_accum=GRAD_ACCUM,
)

In [ ]:
config = dict(model='gpt2_ar', pretrained=PRETRAINED, seed=SEED, n_steps=N_STEPS, step_hours=STEP_HOURS,
              context_length=CONTEXT_LENGTH, batch_size=BATCH_SIZE, grad_accum=GRAD_ACCUM, learning_rate=LEARNING_RATE,
              pretrain_epochs=PRETRAIN_EPOCHS, pretrain_ckpt=str(PRETRAIN_CKPT))
save_results(run_name, test_metrics, test_pred, history, config)
history

### Which events drive the risk?
The classifier weights over the vocabulary are directly interpretable. These are the predicted next events whose logits push the risk up the most.

In [ ]:
import pandas as pd

item_names = {}
for rel, name_col in [('icu/d_items', 'abbreviation'), ('hosp/d_labitems', 'label')]:
    for ext in ['.csv.gz', '.csv']:
        path = MIMIC_DIR / f'{rel}{ext}'
        if path.exists():
            d = pd.read_csv(path, usecols=['itemid', name_col])
            item_names.update(dict(zip(d['itemid'].astype(str), d[name_col])))
            break

weights = ar_model.classifier.weight.detach().float().cpu().numpy().ravel()
vocab = tokenizer.convert_ids_to_tokens(list(range(len(weights))))
is_event = np.array([t[0].isdigit() or t[0] == '-' for t in vocab])  # skip special / demographic tokens
event_ids = np.where(is_event)[0]
for i in event_ids[np.argsort(-weights[event_ids])][:20]:
    item = vocab[i].split('|')[0]
    print(f'{weights[i]:+.4f}  {vocab[i]:<35} {item_names.get(item, "")}')